# Phân tích khám phá dữ liệu (EDA) - CDK

Phân tích dữ liệu benchmark `llm-speed-runs` với mục tiêu dự đoán biến `decode_tps`.

**Nguyên tắc thực hiện:**
- Chỉ đọc dữ liệu gốc, không chỉnh sửa file raw.
- Các phép làm sạch và phân tích chỉ thực hiện trong bộ nhớ (in-memory).
- Không loại bỏ hàng loạt outlier một cách tùy tiện.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

raw_path = Path("C:\\Users\\Admin\\Desktop\\CSI\\SPCK\\data\\raw\\llm-speed-runs-2026-10-22.6.csv")
df = pd.read_csv(raw_path)

print("Loaded:", raw_path)
print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()


In [ ]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("\nDtypes:")
print(df.dtypes)
print("\nBasic missing-value audit:")
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({"missing": missing, "percent": missing_pct})
print(missing_table[missing_table["missing"] > 0])


In [ ]:
overview = pd.DataFrame({
    "metric": ["rows", "columns", "unique run_id", "unique model",
               "unique hardware", "unique runtime", "unique workload",
               "unique source", "unique quant"],
    "value": [len(df), df.shape[1], df["run_id"].nunique(),
              df["model"].nunique(), df["hardware"].nunique(),
              df["runtime"].nunique(), df["workload"].nunique(),
              df["source"].nunique(), df["quant"].nunique()]
})
print(overview.to_string(index=False))

print("\nSource distribution:")
print(df["source"].value_counts())
print("\nRuntime distribution:")
print(df["runtime"].value_counts())
print("\nWorkload distribution:")
print(df["workload"].value_counts())


In [ ]:
print(df["decode_tps"].describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["decode_tps"], bins=30, edgecolor="black")
axes[0].set_title("Histogram of decode_tps")
axes[0].set_xlabel("decode_tps (tokens/s)")
axes[0].set_ylabel("count")

axes[1].boxplot(df["decode_tps"], vert=True)
axes[1].set_title("Boxplot of decode_tps")
axes[1].set_ylabel("decode_tps (tokens/s)")

plt.tight_layout()
plt.show()


In [ ]:
by_runtime = df.groupby("runtime")["decode_tps"].agg(
    count="count", mean="mean", median="median", min="min", max="max"
).round(2)
print("decode_tps by runtime:")
print(by_runtime)

by_workload = df.groupby("workload")["decode_tps"].agg(
    count="count", mean="mean", median="median", min="min", max="max"
).round(2)
print("\ndecode_tps by workload:")
print(by_workload)

by_source = df.groupby("source")["decode_tps"].agg(
    count="count", mean="mean", median="median", min="min", max="max"
).round(2)
print("\ndecode_tps by source:")
print(by_source)

fig, ax = plt.subplots(figsize=(8, 4))
plot_workload = by_workload["mean"].sort_values(ascending=False)
ax.bar(plot_workload.index, plot_workload.values, edgecolor="black")
ax.set_title("decode_tps trung bình theo workload")
ax.set_xlabel("Workload")
ax.set_ylabel("decode_tps trung bình (token/s)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
top_hardware = df.groupby("hardware")["decode_tps"].agg(
    count="count", mean="mean"
).sort_values("count", ascending=False).head(12).round(2)
print("Top hardware by benchmark count:")
print(top_hardware)

print("\nSource x runtime:")
print(pd.crosstab(df["source"], df["runtime"]))


In [ ]:
run_sizes = df["run_id"].value_counts()
print("run_id group size summary:")
print(run_sizes.describe().round(3))
print("\nNumber of run_id with more than one row:", (run_sizes > 1).sum(), "of", len(run_sizes))

multi_runs = run_sizes[run_sizes > 1].index
multi_df = df[df["run_id"].isin(multi_runs)]
variation = multi_df.groupby("run_id").agg(
    n_rows=("run_id", "size"),
    distinct_workload=("workload", "nunique"),
    distinct_runtime=("runtime", "nunique"),
    distinct_hardware=("hardware", "nunique"),
    distinct_model=("model", "nunique"),
).round(2)
print("\nMulti-row run_id variation summary:")
print(variation.describe().round(2))
print("\nExample repeated run_id:")
print(df[df["run_id"] == multi_runs[0]].to_string(index=False))


In [ ]:
print("Duplicate exact rows:", df.duplicated().sum())
print("Duplicate run_id + workload rows:", df.duplicated(subset=["run_id", "workload"]).sum())

model_size_parse = df["model_size"].astype("string").str.extract(r"^\s*([0-9]+(?:\.[0-9]+)?)\s*([A-Za-z]+)?")
print("\nParsed model_size examples:")
print(model_size_parse.value_counts(dropna=False).head(10))

print("\nQuant coverage:")
print(df["quant"].value_counts(dropna=False).head(15))


In [ ]:
# Exploratory cleaning, only in-memory. Raw file is not modified.
df_clean = df.copy()

# Add a numeric model size in billions for later modeling experiments.
df_clean["model_size_b"] = (
    df_clean["model_size"].astype("string")
    .str.extract(r"^\s*([0-9]+(?:\.[0-9]+)?)", expand=False)
    .astype("float")
)

# Preserve rows; only document missing categories.
print("Exploratory copy shape:", df_clean.shape)
print("Unavailable model_size_b:", df_clean["model_size_b"].isna().sum())
print("Quant missing:", df_clean["quant"].isna().sum())
print("Target missing:", df_clean["decode_tps"].isna().sum())

# Candidate columns for future Speed Prediction model.
candidate_X = ["model", "hardware", "runtime", "workload", "quant",
               "prompt_tokens", "output_tokens", "batch_size"]
candidate_y = "decode_tps"
print("\nCandidate X:", candidate_X)
print("Candidate y:", candidate_y)
print("Rows available for complete-case candidate subset:",
      df_clean[candidate_X + [candidate_y]].dropna().shape[0])


## Kết quả quan sát, giả thuyết và hạn chế

### Kết quả quan sát dựa trên dữ liệu gốc
- Tập dữ liệu có 751 hàng benchmark, 18 cột, 335 `run_id` khác nhau, 106 mô hình khác nhau.
- `source` được chia thành `community` (408 hàng) và `llm-speed` (343 hàng).
- `decode_tps` có đầy đủ cho tất cả các hàng, giá trị dao động rộng (khoảng 3,76 đến 466,95 token/s).
- `model_size` bị thiếu khoảng 84,55% số hàng, không thể dùng làm đặc trưng chính.
- `quant` bị thiếu khoảng 36,35% số hàng; phần lớn sử dụng runtime `llama.cpp`.
- Các nhóm phần cứng phổ biến nhất: RTX 5090/4090/3090 và Apple Silicon; mức độ phủ sóng không đồng đều.
- Các hàng có cùng `run_id` không trùng lặp hoàn toàn; cùng một run thường chỉ thay đổi `workload`, còn hardware/model/runtime giữ ổn định.
- `batch_size` có giá trị không đổi (bằng 1) trong tập dữ liệu này.

### Giả thuyết (chưa được xác minh)
- Nhóm phần cứng (hardware) và kích thước mô hình (`model_size`) có thể giải thích phần lớn biến thiên của `decode_tps`.
- `quant` và runtime có thể tạo ra sự khác biệt về tốc độ sau khi kiểm soát hardware và mô hình.
- Sự khác biệt giữa các workload có thể chủ yếu phản ánh độ dài prompt/output chứ không phải tốc độ sinh thực tế.

### Hạn chế
- `hardware` được lưu dưới dạng chuỗi mô tả tự do, không có thông số cấu trúc (CPU/GPU/RAM/VRAM).
- Dữ liệu kết hợp giữa cộng đồng và nền tảng llm-speed, có thể tồn tại thiên vị theo nguồn dữ liệu.
- Các workload, quantization, runtime không được cân bằng.
- Tập dữ liệu phù hợp để nghiên cứu tốc độ dưới điều kiện benchmark, nhưng không đủ để dự đoán chính xác cho mọi máy tính người dùng mà không có đánh giá khả năng tương thích phần cứng.
- Chưa loại bỏ outlier; các giá trị cực trị có thể là kết quả benchmark thực tế hoặc lỗi nhập liệu và cần xem xét cẩn thận.


## Các bước tiếp theo trước khi xây dựng mô hình hồi quy tuyến tính

1. Quyết định mô hình hóa toàn bộ dữ liệu hay phân tầng theo hardware/runtime.
2. Xây dựng quy tắc làm sạch có thể tái lập cho các giá trị thiếu của `quant` và `model_size`.
3. Tạo các đặc trưng số cho phần cứng nếu có thể đưa ra giả định đáng tin cậy và có tài liệu hóa.
4. Giữ `run_id + workload` làm khóa duy nhất an toàn hơn.
5. Sử dụng phương pháp chia train/test và báo cáo các chỉ số hồi quy trong notebook tiếp theo.


## Numeric features: variance & unique values

Kiểm tra `prompt_tokens` và `batch_size` theo yêu cầu.

In [ ]:
print("prompt_tokens unique:", df["prompt_tokens"].nunique())
print("prompt_tokens variance:", df["prompt_tokens"].var())
print("\nbatch_size unique:", df["batch_size"].nunique())
print("batch_size value counts:")
print(df["batch_size"].value_counts())
print("\n>> batch_size is near-constant (or constant). Do NOT force batch_size into scatter/correlation plots per requirements.")

## Scatter: prompt_tokens vs decode_tps

Kiểm tra mối quan hệ đơn biến giữa `prompt_tokens` và `decode_tps`. Lưu ý: chỉ áp dụng Pearson cho numeric có đủ variation.

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df["prompt_tokens"], df["decode_tps"], alpha=0.5)
ax.set_title("Biểu đồ phân tán: prompt_tokens vs decode_tps")
ax.set_xlabel("prompt_tokens (số token)")
ax.set_ylabel("decode_tps (token/s)")
plt.tight_layout()
plt.show()

num_check = df[["prompt_tokens", "decode_tps"]].dropna()
if num_check["prompt_tokens"].nunique() > 1:
    corr_pt = num_check[["prompt_tokens", "decode_tps"]].corr(method="pearson")
    print("Pearson correlation (prompt_tokens, decode_tps):")
    print(corr_pt)
else:
    print("prompt_tokens has insufficient variation for Pearson.")

## Correlation heatmap (numeric only)

Chỉ tính Pearson cho numeric có đủ variation. Không áp dụng trực tiếp cho categorical.

In [ ]:
candidate_num = []
for c in ["prompt_tokens", "batch_size"]:
    if c in df.columns and df[c].nunique() > 1:
        candidate_num.append(c)

if len(candidate_num) >= 1:
    corr_df = df[candidate_num + ["decode_tps"]].corr(method="pearson")
    print("Pearson correlation matrix:")
    print(corr_df)
    fig, ax = plt.subplots(figsize=(6,4))
    im = ax.imshow(corr_df.values, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr_df.columns)))
    ax.set_yticks(range(len(corr_df.index)))
    ax.set_xticklabels(corr_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr_df.index)
    plt.colorbar(im)
    ax.set_title("Biểu đồ tương quan (Pearson) - chỉ biến số")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric features with sufficient variation for a meaningful correlation heatmap.")
    print("Note: batch_size is constant (or near-constant) - skipped as requested.")